In [5]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Abrir sesión de Spark
spark = SparkSession.builder.appName("Ejemplo").getOrCreate()

df = pd.DataFrame({
	"id_usuarios": range(1, 21),
	"ventas": [120, 85, 230, 150, 95, 310, 180, 75, 260, 140,
			   200, 110, 325, 90, 175, 240, 130, 285, 160, 105],
	"cuidad": ["Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao",
			   "Málaga", "Alicante", "Granada", "Zaragoza", "Murcia",
			   "Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao",
			   "Málaga", "Alicante", "Granada", "Zaragoza", "Murcia"]
})

df_spark = spark.createDataFrame(df)
df_spark.show()

# Abrir tabla temporal para posteriormente atacar con consulta SQL
df_spark.createOrReplaceTempView("ventas_usuarios")

# Ejemplo de consulta SQL sobre la tabla temporal
resultado = spark.sql("SELECT cuidad, SUM(ventas) AS total_ventas FROM ventas_usuarios GROUP BY cuidad")
resultado.show()

# Uso de API de DataFrame de Spark
resultado_api = df_spark.groupBy("cuidad").sum("ventas").withColumnRenamed("sum(ventas)", "total_ventas")
resultado_api.show()

# Genera otro dataframe que tenga sentido para practicar cruces con diferentes tipos de JOIN (right, left, inner, outer, ...)
df_usuarios = pd.DataFrame({
	"id_usuarios": range(1, 21),
	"nombre": ["Usuario" + str(i) for i in range(1, 21)],
	"edad": [25, 30, 22, 35, 28, 40, 32, 27, 33, 29,
			31, 26, 34, 23, 36, 38, 24, 37, 39, 21]
})

df_usuarios_spark = spark.createDataFrame(df_usuarios)
df_usuarios_spark.show()

# Abrir tabla temporal para posteriormente atacar con consulta SQL
df_usuarios_spark.createOrReplaceTempView("usuarios")

# Ejemplo de diferentes tipos de JOIN entre las dos tablas temporales
resultado_inner = spark.sql("""
    SELECT v.cuidad, v.ventas, u.nombre, u.edad
    FROM ventas_usuarios v
    INNER JOIN usuarios u ON v.id_usuarios = u.id_usuarios
""")
resultado_inner.show()

resultado_left = spark.sql("""
    SELECT v.cuidad, v.ventas, u.nombre, u.edad
    FROM ventas_usuarios v
    LEFT JOIN usuarios u ON v.id_usuarios = u.id_usuarios
""")
resultado_left.show()

resultado_right = spark.sql("""
    SELECT v.cuidad, v.ventas, u.nombre, u.edad
    FROM ventas_usuarios v
    RIGHT JOIN usuarios u ON v.id_usuarios = u.id_usuarios
""")
resultado_right.show()

resultado_outer = spark.sql("""
    SELECT v.cuidad, v.ventas, u.nombre, u.edad
    FROM ventas_usuarios v
    FULL OUTER JOIN usuarios u ON v.id_usuarios = u.id_usuarios
""")
resultado_outer.show()

+-----------+------+---------+
|id_usuarios|ventas|   cuidad|
+-----------+------+---------+
|          1|   120|   Madrid|
|          2|    85|Barcelona|
|          3|   230| Valencia|
|          4|   150|  Sevilla|
|          5|    95|   Bilbao|
|          6|   310|   Málaga|
|          7|   180| Alicante|
|          8|    75|  Granada|
|          9|   260| Zaragoza|
|         10|   140|   Murcia|
|         11|   200|   Madrid|
|         12|   110|Barcelona|
|         13|   325| Valencia|
|         14|    90|  Sevilla|
|         15|   175|   Bilbao|
|         16|   240|   Málaga|
|         17|   130| Alicante|
|         18|   285|  Granada|
|         19|   160| Zaragoza|
|         20|   105|   Murcia|
+-----------+------+---------+

+---------+------------+
|   cuidad|total_ventas|
+---------+------------+
|   Madrid|         320|
|Barcelona|         195|
| Valencia|         555|
|   Bilbao|         270|
|  Sevilla|         240|
|   Málaga|         550|
|  Granada|         360|
| Ali